# RobustModelMaker Benchmark Suite

This notebook runs three real scientific benchmarks to demonstrate that **RobustModelMaker (ROBUST)** achieves comparable predictive performance to a full-feature baseline while selecting a substantially smaller feature subset.

Each scenario:
1. Loads a real scientific dataset via BenchMake archetypal train/test split (adversarial by design — conservative scores).
2. Fits RobustModelMaker on the training partition (bootstrap stability selection + nested CV).
3. Fits a full-feature nested-CV **baseline** on the same training partition with the same algorithm.
4. Runs three **comparator** feature selectors on the same outer-fold structure: **ANOVA/SelectKBest**, **RFECV**, and **Boruta**.
5. Reports **two numbers per method per dataset**: predictive score (AUC or RMSE) and **Jaccard stability** (mean pairwise similarity of selected feature sets across the 10 outer folds).
6. Applies a 25+ test statistical battery to the RobustModelMaker vs baseline per-fold score vectors.
7. Classifies the ROBUST outcome as **preserved** (no significant loss), **sig. better \***, or **sig. worse \***.

All methods use **Random Forest (RF)** as the underlying estimator, making the comparison a clean demonstration of stability selection against other feature-selection strategies across all task types:
- **SECOM Manufacturing** — binary classification (590 features, real NaNs)
- **Urban Land Cover** — 9-class multiclass (147 aerial-image features)
- **Graphene Oxide Bulk** — regression (~309 MD structural descriptors)

Boruta requires `pip install boruta`; if not installed it is skipped gracefully. Run cells individually to inspect each scenario, or run all to see the multi-method comparison table at the bottom.

## Setup

The cells below load `benchmark_suite.py` as a module using `importlib` so that the `if __name__ == "__main__":` block does **not** execute automatically.  Adjust `REPO_ROOT` if your directory layout differs.

### Per-dataset stability thresholds

`stability_threshold` controls how often a feature must appear across bootstrap resamples to pass selection.  The optimal value differs by dataset — find it by running **`tools/Threshold_Optimisation.ipynb`**.

The setup cell imports `THRESHOLD_OVERRIDES` from `benchmark_suite.py`.  A **threshold configuration cell immediately below the setup cell** is the single place to paste your optimiser results:

```python
THRESHOLD_OVERRIDES.update({
    "SECOM Manufacturing":  0.60,   # composite-optimal (full ThresholdOptimizer sweep)
    "Urban Land Cover":     0.80,
    "Graphene Oxide Bulk":  0.75,   # set once you have a value from the optimiser
})
```

The effective threshold for each dataset is printed at the start of `run_scenario()`.  Setting a value to `None` falls back to the global `ROBUST_PARAMS['stability_threshold']`.

**Priority** (lowest → highest): `ROBUST_PARAMS['stability_threshold']` → `THRESHOLD_OVERRIDES[dataset_name]` → `ds.robust_params_override`

In [3]:
import sys
import os
import importlib.util
import warnings
from pathlib import Path
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Path configuration ────────────────────────────────────────────────────
# Set REPO_ROOT to the directory containing RobustModelMaker.py.
# Uncomment and edit the line below if running from a different location.
# REPO_ROOT = Path(r"path_to_your_RobustModelMaker_directory")
REPO_ROOT = Path(r"C:\Users\Amanda\Favorites\Machine Learning\RobustModelMaker")
sys.path.insert(0, str(REPO_ROOT))

# ── Load benchmark_suite as a module ─────────────────────────────────────
# importlib prevents the __main__ block from running; each scenario is
# controlled individually in the cells below.
_bs_path = REPO_ROOT / "benchmarks" / "benchmark_suite.py"
if not _bs_path.exists():
    raise FileNotFoundError(f"benchmark_suite.py not found at {_bs_path}")

_spec = importlib.util.spec_from_file_location("benchmark_suite", str(_bs_path))
bs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(bs)

# ── Convenience aliases: dataset loaders ─────────────────────────────────
_load_secom            = bs._load_secom
_load_urban_land_cover = bs._load_urban_land_cover
_load_graphene_oxide   = bs._load_graphene_oxide
DataUnavailable        = bs.DataUnavailable

# ── Convenience aliases: runners and summary tables ───────────────────────
run_scenario             = bs.run_scenario
print_summary_table      = bs.print_summary_table
print_comparator_summary = bs.print_comparator_summary   # multi-method pivot table

# ── Convenience aliases: comparator utilities ────────────────────────────
jaccard_stability    = bs.jaccard_stability    # mean pairwise Jaccard across fold sets
ComparatorResult     = bs.ComparatorResult     # result container for one comparator
run_anova_nested_cv  = bs.run_anova_nested_cv  # ANOVA/SelectKBest comparator
run_rfecv_nested_cv  = bs.run_rfecv_nested_cv  # RFECV comparator
run_boruta_nested_cv = bs.run_boruta_nested_cv # Boruta comparator (None if not installed)

ROBUST_PARAMS      = bs.ROBUST_PARAMS
THRESHOLD_OVERRIDES = bs.THRESHOLD_OVERRIDES   # per-dataset thresholds — edit in next cell

# ── Global parameter summary ─────────────────────────────────────────────
p = ROBUST_PARAMS
print("Benchmark suite loaded successfully.")
print(f"  outer_cv={p['outer_cv']}  inner_cv={p['inner_cv']}  "
      f"n_bootstrap={p['n_bootstrap']}  n_iter={p['n_iter']}  "
      f"random_state={p['random_state']}")
print(f"  global stability_threshold={p['stability_threshold']}  "
      f"(overridden per dataset — see threshold config cell below)")
print()
print("Comparators (run automatically inside run_scenario):")
print("  • ANOVA/SelectKBest  k = max(10, p//10)  ~10% retention    [sklearn — no extra install]")
print("  • RFECV              step = ⌊√p⌋, cv = 5                  [sklearn — no extra install]")
print("  • Boruta             n=100, d=7, perc=100  ~15–40% ret.    [pip install boruta]")

# Containers for cross-scenario results (populated by the benchmark cells below)
result_secom    = 0.60
result_urban    = 0.80
result_graphene = 0.75

CuPy not installed. Using NumPy for CPU computations.
Benchmark suite loaded successfully.
  outer_cv=5  inner_cv=5  n_bootstrap=20  n_iter=20  random_state=42
  global stability_threshold=0.75  (overridden per dataset — see threshold config cell below)

Comparators (run automatically inside run_scenario):
  • ANOVA/SelectKBest  k = max(10, p//10)  ~10% retention    [sklearn — no extra install]
  • RFECV              step = ⌊√p⌋, cv = 5                  [sklearn — no extra install]
  • Boruta             n=100, d=7, perc=100  ~15–40% ret.    [pip install boruta]


In [ ]:
# ----------------------------------------------------------------------------
# THRESHOLD CONFIGURATION
# ----------------------------------------------------------------------------
# Edit these values after running tools/Threshold_Optimisation.ipynb,
# then re-run this cell.  Set any value to None to fall back to the global
# default in ROBUST_PARAMS['stability_threshold'].

THRESHOLD_OVERRIDES.update({
    'SECOM Manufacturing':   0.60,   # composite-optimal (full ThresholdOptimizer sweep)
    'Urban Land Cover':      0.80,
    'Graphene Oxide Bulk':   0.75,
    'Synthetic Binary':      0.70,   # synthetic: default works well at K=10/B=100
    'Synthetic Multiclass':  0.70,
    'Synthetic Regression':  0.70,
})

print('Effective stability_threshold per dataset  (global default = '
      f"{ROBUST_PARAMS[\"stability_threshold\"]}):\n")
for ds_name in ['SECOM Manufacturing', 'Urban Land Cover', 'Graphene Oxide Bulk',
                'Synthetic Binary', 'Synthetic Multiclass', 'Synthetic Regression']:
    val = THRESHOLD_OVERRIDES.get(ds_name)
    label = 'ThresholdOptimizer' if val is not None else 'global default'
    eff   = val if val is not None else ROBUST_PARAMS['stability_threshold']
    mark  = '*' if val is not None else 'o'
    print(f'  {mark}  {ds_name:<28} thr = {eff:.2f}  [{label}]')


---
## Benchmark 1: SECOM Manufacturing (binary classification)

| Property | Value |
|---|---|
| Dataset | SECOM semiconductor sensor data (UCI) |
| Samples x features | 1,567 x 590 |
| Task | Binary pass/fail (~7% failure rate) |
| Algorithm | Random Forest (RF) |
| Metric | AUC-ROC (higher = better) |
| Challenge | Heavy class imbalance, extensive real NaN values |

The dataset is split with BenchMake's archetypal partitioning (adversarial: train and test sets are maximally separated in feature space). RobustModelMaker runs nested CV on the training partition only; the held-out test set is not used during fitting.

In [6]:
# Load SECOM dataset (downloads from UCI if not cached; requires internet access)
try:
    ds_secom = _load_secom()
    print(f"SECOM loaded: {ds_secom.X.shape[0]} samples x {ds_secom.X.shape[1]} features")
    print(f"  Train: {ds_secom.X_train.shape[0]}  |  Test (held-out): {ds_secom.X_test.shape[0]}")
except DataUnavailable as e:
    print(f"SKIP: {e}")
    ds_secom = None

SECOM loaded: 1567 samples x 590 features
  Train: 1253  |  Test (held-out): 314


In [7]:
if ds_secom is not None:
    result_secom = run_scenario(ds_secom, verbose=True)
else:
    print("SECOM dataset unavailable -- skipped.")


>>> SECOM Manufacturing: running ROBUST(RF, binary, thr=0.60) ...
    threshold from THRESHOLD_OVERRIDES  (global default=0.75)
    ROBUST done (308 features, Jaccard stability=0.660). Running baseline ...
    Baseline done. Running ANOVA comparator ...
    ANOVA done (59 features, Jaccard=0.417). Running RFECV ...
    RFECV done (254 features, Jaccard=0.274). Running Boruta ...
    Boruta done (10 features, Jaccard=0.224).

  SCENARIO: SECOM Manufacturing
  Semiconductor process sensor data; 1567 samples x 590 features, binary pass/fail, ~7% failure rate, extensive real NaN values
  Algorithm : RF   Task : binary   Runtime : 2480.1s
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline

                                         DATASET                                        
------------------------------------------------------------------------------------------
  Total : 1567 samples x 590 features  (BenchMake split: train=1253, held

---
## Benchmark 2: Urban Land Cover (9-class multiclass)

| Property | Value |
|---|---|
| Dataset | Urban Land Cover (UCI) |
| Samples x features | 675 x 147 |
| Task | 9 urban land cover classes from aerial image segments |
| Algorithm | Random Forest (RF) |
| Metric | AUC-OVR weighted (higher = better) |
| Challenge | Multi-class structure; highly correlated spectral/texture descriptors |

Classification is segment-level (not pixel-level), so there is no spatial autocorrelation between samples. RobustModelMaker is expected to drop a large fraction of the 147 highly correlated imagery features without significant performance loss.

In [9]:
# Load Urban Land Cover dataset (downloads from UCI; requires internet access)
try:
    ds_urban = _load_urban_land_cover()
    print(f"Urban Land Cover loaded: {ds_urban.X.shape[0]} samples x {ds_urban.X.shape[1]} features")
    print(f"  Train: {ds_urban.X_train.shape[0]}  |  Test (held-out): {ds_urban.X_test.shape[0]}")
    print(f"  Classes: {sorted(ds_urban.y.unique())}")
except DataUnavailable as e:
    print(f"SKIP: {e}")
    ds_urban = None

Urban Land Cover loaded: 675 samples x 147 features
  Train: 540  |  Test (held-out): 135
  Classes: ['asphalt ', 'building ', 'car ', 'concrete ', 'grass ', 'pool ', 'shadow ', 'soil ', 'tree ']


In [10]:
if ds_urban is not None:
    result_urban = run_scenario(ds_urban, verbose=True)
else:
    print("Urban Land Cover dataset unavailable -- skipped.")


>>> Urban Land Cover: running ROBUST(RF, multiclass, thr=0.80) ...
    threshold from THRESHOLD_OVERRIDES  (global default=0.75)
    ROBUST done (56 features, Jaccard stability=0.801). Running baseline ...
    Baseline done. Running ANOVA comparator ...
    ANOVA done (14 features, Jaccard=1.000). Running RFECV ...
    RFECV done (85 features, Jaccard=0.468). Running Boruta ...
    Boruta done (103 features, Jaccard=0.895).

  SCENARIO: Urban Land Cover
  Aerial image segments; 675 samples x 147 spectral/texture features, 9 urban land cover classes, segment-level (no spatial autocorrelation)
  Algorithm : RF   Task : multiclass   Runtime : 991.1s
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline

                                         DATASET                                        
------------------------------------------------------------------------------------------
  Total : 675 samples x 147 features  (BenchMake split: trai

C:\ProgramData\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


---
## Benchmark 3: Graphene Oxide Bulk (regression)

| Property | Value |
|---|---|
| Dataset | CSIRO Graphene Oxide Bulk (local CSV) |
| Samples x features | ~1,617 x ~309 (after NaN-column and constant-column drop) |
| Task | Regression: Formation energy (eV) from structural descriptors |
| Algorithm | Random Forest (RF) |
| Metric | RMSE in eV (lower = better; displayed as positive RMSE) |
| Challenge | Very high-dimensional descriptor space; sparse/correlated MD features |

RF importance scores (MDI variance reduction) are naturally non-uniform across correlated structural descriptors, giving bootstrap stability selection a discriminative frequency distribution without algorithm-specific tuning.

**File requirement:** `Graphene_Oxide_Bulk.csv` must be present in the `benchmarks/` directory. This is a CSIRO dataset and is not downloaded automatically, but more information can be obtained here: https://doi.org/10.25919/5e30b45f9852c 

In [12]:
# Load Graphene Oxide Bulk dataset (requires local CSV file)
try:
    ds_graphene = _load_graphene_oxide()
    print(f"Graphene Oxide loaded: {ds_graphene.X.shape[0]} samples x {ds_graphene.X.shape[1]} features")
    print(f"  Train: {ds_graphene.X_train.shape[0]}  |  Test (held-out): {ds_graphene.X_test.shape[0]}")
    import numpy as np
    y_arr = ds_graphene.y.to_numpy(dtype=float)
    print(f"  Formation_energy: min={y_arr.min():.2f} eV  max={y_arr.max():.2f} eV  mean={y_arr.mean():.2f} eV")
    print(f"  Dataset-specific overrides: {ds_graphene.robust_params_override}")
except DataUnavailable as e:
    print(f"SKIP: {e}")
    ds_graphene = None

Graphene Oxide loaded: 1617 samples x 309 features
  Train: 1293  |  Test (held-out): 324
  Formation_energy: min=-88.28 eV  max=-66.09 eV  mean=-80.42 eV
  Dataset-specific overrides: {}


In [13]:
if ds_graphene is not None:
    result_graphene = run_scenario(ds_graphene, verbose=True)
else:
    print("Graphene Oxide dataset unavailable -- skipped.")


>>> Graphene Oxide Bulk: running ROBUST(RF, regression, thr=0.75) ...
    threshold from THRESHOLD_OVERRIDES  (global default=0.75)
    ROBUST done (138 features, Jaccard stability=0.899). Running baseline ...
    Baseline done. Running ANOVA comparator ...
    ANOVA done (30 features, Jaccard=1.000). Running RFECV ...
    RFECV done (3 features, Jaccard=0.520). Running Boruta ...
    Boruta done (115 features, Jaccard=0.958).

  SCENARIO: Graphene Oxide Bulk
  MD-derived structural descriptors; 1617 samples x 309 features, regression target = Formation_energy (eV), 19 stoichiometries
  Algorithm : RF   Task : regression   Runtime : 13873.1s
  Legend: ROBUST = RobustModelMaker stability-selected subset  |  BL = Full-feature nested-CV baseline

                                         DATASET                                        
------------------------------------------------------------------------------------------
  Total : 1617 samples x 309 features  (BenchMake split: train=12

---
## Cross-scenario summary

The cells below consolidate key figures from all three benchmarks across **five methods**: RobustModelMaker, full-feature Baseline, ANOVA, RFECV, and Boruta.

Two tables are printed:

1. **RobustModelMaker vs Baseline** — original compact summary with feature counts, scores, Jaccard stability for RobustModelMaker, and statistical outcome (Wilcoxon p-value).
2. **Multi-method comparison** (`print_comparator_summary`) — pivot showing Score ± Std and Jaccard stability for every method × dataset.

Metric conventions:
- Regression rows show **RMSE** (positive, lower is better).
- Classification rows show **AUC** (higher is better).
- **Stability** = mean pairwise Jaccard of selected feature sets across the 10 outer folds (0 = disjoint fold-to-fold, 1 = identical every fold). Baseline has no selection, so its Jaccard is n/a by construction.
- **+delta** for RobustModelMaker is positive when RobustModelMaker is better than baseline in each metric's own convention.

> **Note on BenchMake splits:** All benchmarks use adversarial archetypal splits that maximise distance between train and test sets. Scores are conservative relative to conventional random splits; the comparison across methods is internally consistent because all use the same split.

In [ ]:
# Include synthetic scenarios in the cross-scenario tables once they have been run.
# Variables that have not been assigned yet are gracefully skipped.
_synth = []
for _v in ('result_synb', 'result_synm', 'result_synr'):
    _synth.append(globals().get(_v))

all_results = [r for r in [result_secom, result_urban, result_graphene, *_synth] if r is not None]

if all_results:
    # 1. ROBUST vs baseline compact table (now includes Jaccard stability column)
    print_summary_table(all_results)
    print()
    # 2. Multi-method pivot: Score ± Std and Jaccard stability for every method × dataset
    print_comparator_summary(all_results)
    total = sum(r['elapsed'] for r in all_results)
    print(f'\n  Total wall-clock time: {total:.1f}s')
else:
    print('No results to summarise.')


---
## Comparator deep-dive

The table below repeats the **COMPARATOR COMPARISON** block from each scenario's verbose output, with an added **Outcome vs BL** column derived from a paired Wilcoxon signed-rank test on the per-fold scores of each method against the full-feature baseline. This is the same convention used for the ROBUST-vs-baseline outcome label printed by `run_scenario`, applied uniformly to every method. The function `print_scenario_comparators` operates on the in-memory result dicts, so re-running this cell after editing `benchmark_suite.py` does not require re-fitting any models.

| Method | Selection mechanism | Key hyperparameters | Approx. retention |
|---|---|---|---|
| **RobustModelMaker** | Bootstrap stability selection | `threshold=N`, `n_bootstrap=20` | data-driven |
| **Baseline** | None (reference) | all features every fold | 100% |
| **ANOVA** | Univariate F-test filter | `k = max(10, p//10)` (~10%) | ~10% |
| **RFECV** | Recursive elimination + CV elbow | `step = floor(sqrt(p))`, `cv = 5` | data-driven |
| **Boruta** | Shadow-feature binomial test | `n=100`, `d=7`, `perc=100` | data-driven |

Outcome labels follow the framework's standard convention: `preserved` (paired test p >= 0.05, no significant difference from baseline; the primary success criterion for ROBUST), `sig. better *` (p < 0.05 with positive delta), or `sig. worse *` (p < 0.05 with negative delta). The Baseline row is the reference and has no outcome of its own.


In [ ]:
# Per-dataset comparator comparison, including 'Outcome vs BL' from paired Wilcoxon
# signed-rank test on per-fold scores. Includes synthetic scenarios once they are run.

_scenarios = [
    ('SECOM Manufacturing',  result_secom),
    ('Urban Land Cover',     result_urban),
    ('Graphene Oxide Bulk',  result_graphene),
    ('Synthetic Binary',     globals().get('result_synb')),
    ('Synthetic Multiclass', globals().get('result_synm')),
    ('Synthetic Regression', globals().get('result_synr')),
]

for _name, _r in _scenarios:
    if _r is None:
        print(f'\n{_name}: not yet run or unavailable.')
        continue
    bs.print_scenario_comparators(_r)


---
## Stability analysis

**Jaccard stability** quantifies how consistent each method's feature selection is across the 10 outer folds:

$$J = \frac{1}{\binom{K}{2}} \sum_{i < j} \frac{|S_i \cap S_j|}{|S_i \cup S_j|}$$

where $S_i$ is the set of features selected in outer fold $i$. A value of **1.0** means every fold produced the identical feature set; **0.0** means every pair of fold selections is entirely disjoint.

### Expected patterns across methods

| Method | What drives stability | Expected behaviour |
|---|---|---|
| **RobustModelMaker** | Bootstrap frequency threshold (0.75) filters noise features | High stability when signal dominates; by construction ROBUST is optimised for this |
| **Baseline** | No selection — same features every fold | n/a (identical set every fold) |
| **ANOVA** | Univariate F-statistic ranking fixed to k features | High when main effects are stable across folds; can drop if the top-k frontier shifts |
| **RFECV** | CV-curve elbow determines k per fold | Moderate; depends on how sharp and consistent the elbow is |
| **Boruta** | Randomised shadow-feature binomial test | Moderate to high; bounded by shadow-feature stochasticity (`n_estimators=100` mitigates this) |

The three pivot tables below show stability, mean feature count, and dimensionality reduction for each method × dataset combination.

In [19]:
import pandas as pd
import numpy as np

# ── Jaccard stability pivot tables ────────────────────────────────────────
_rows = []
for _ds, _r in [("SECOM", result_secom), ("Urban", result_urban), ("Graphene", result_graphene)]:
    if _r is None:
        continue
    _n_bl = _r["n_features_bl"]
    _rows.append({"Dataset": _ds, "Method": "ROBUST",
                  "Stability": _r.get("robust_stability", float("nan")),
                  "Mean feats": float(_r["n_features_robust"]), "Total": float(_n_bl)})
    _rows.append({"Dataset": _ds, "Method": "Baseline",
                  "Stability": float("nan"),
                  "Mean feats": float(_n_bl), "Total": float(_n_bl)})
    for _key, _cres in (_r.get("comparators") or {}).items():
        if _cres is None:
            continue
        _rows.append({"Dataset": _ds, "Method": _cres.name,
                      "Stability": _cres.stability,
                      "Mean feats": _cres.mean_n_features, "Total": float(_n_bl)})

if _rows:
    _df = pd.DataFrame(_rows)

    print("── Jaccard stability  (higher = more consistent feature selection across folds) ──\n")
    _piv_stab = (_df.pivot(index="Method", columns="Dataset", values="Stability")
                    .rename_axis(None, axis=1).round(3))
    print(_piv_stab.to_string())

    print("\n── Mean features selected per fold ──\n")
    _piv_feat = (_df.pivot(index="Method", columns="Dataset", values="Mean feats")
                    .rename_axis(None, axis=1).round(0))
    # Display as integers; NaN stays as NaN (pandas handles NaN outside float_format)
    print(_piv_feat.to_string(float_format=lambda x: f'{int(x)}'))

    print("\n── Reduction from full feature space (%) ──\n")
    _df["Reduction %"] = (1.0 - _df["Mean feats"] / _df["Total"]) * 100.0
    _piv_red = (_df.pivot(index="Method", columns="Dataset", values="Reduction %")
                   .rename_axis(None, axis=1).round(1))
    print(_piv_red.to_string())
else:
    print("No results yet — run the benchmark cells above first.")

── Jaccard stability  (higher = more consistent feature selection across folds) ──

                        Graphene  SECOM  Urban
Method                                        
ANOVA k=14                   NaN    NaN  1.000
ANOVA k=30                 1.000    NaN    NaN
ANOVA k=59                   NaN  0.417    NaN
Baseline                     NaN    NaN    NaN
Boruta n=100 d=7 p=100     0.958  0.224  0.895
RFECV step=12                NaN    NaN  0.468
RFECV step=17              0.520    NaN    NaN
RFECV step=24                NaN  0.274    NaN
ROBUST                     0.899  0.660  0.801

── Mean features selected per fold ──

                        Graphene  SECOM  Urban
Method                                        
ANOVA k=14                   NaN    NaN     14
ANOVA k=30                    30    NaN    NaN
ANOVA k=59                   NaN     59    NaN
Baseline                     309    590    147
Boruta n=100 d=7 p=100       115     10    103
RFECV step=12                N

---
## Inline unit tests

The cell below verifies `jaccard_stability()` on hand-computed edge cases, checks `ComparatorResult` attribute invariants, and — once the benchmark cells above have been executed — runs smoke tests against the live results. These parallel the `TestComparators` pytest class in `benchmark_suite.py` and can be re-run independently without re-fitting any models.

In [21]:
import numpy as np

# ── Jaccard stability unit tests ──────────────────────────────────────────
_PASS = "✔"
_FAIL = "✘"

def _check(name, condition):
    status = _PASS if condition else _FAIL
    print(f"  {status}  {name}")
    if not condition:
        raise AssertionError(f"FAILED: {name}")

print("=== jaccard_stability() unit tests ===\n")

_check("identical sets  → J = 1.0",
       abs(jaccard_stability([{"a", "b", "c"}] * 5) - 1.0) < 1e-9)

_check("disjoint sets   → J = 0.0",
       abs(jaccard_stability([{"a", "b"}, {"c", "d"}, {"e", "f"}]) - 0.0) < 1e-9)

_check("50% overlap     → J = 0.5",
       abs(jaccard_stability([{"a", "b", "c"}, {"b", "c", "d"}]) - 0.5) < 1e-9)

# {a,b}∩{a,c}=1/3  {a,b}∩{b,c}=1/3  {a,c}∩{b,c}=1/3  → mean = 1/3
_check("three-way 1/3   → J ≈ 1/3",
       abs(jaccard_stability([{"a", "b"}, {"a", "c"}, {"b", "c"}]) - 1 / 3) < 1e-9)

_check("single set      → NaN",     np.isnan(jaccard_stability([{"a", "b"}])))
_check("empty list      → NaN",     np.isnan(jaccard_stability([])))
_check("two empty sets  → J = 1.0", abs(jaccard_stability([set(), set()]) - 1.0) < 1e-9)

# ── ComparatorResult structural tests ─────────────────────────────────────
print("\n=== ComparatorResult structural tests ===\n")

_scores = np.array([0.80, 0.85, 0.82, 0.79, 0.83])
_sets   = [{"a", "b"}, {"a", "c"}, {"b", "c"}, {"a", "b"}, {"b", "c"}]
_cr     = ComparatorResult("test", _scores, _sets, {"k": 2})

_check("mean_score correct",    abs(_cr.mean_score     - _scores.mean()) < 1e-9)
_check("std_score correct",     abs(_cr.std_score      - _scores.std())  < 1e-9)
_check("mean_n_features = 2.0", abs(_cr.mean_n_features - 2.0)           < 1e-9)
_check("stability in [0, 1]",   0.0 <= _cr.stability <= 1.0)

# ── Post-run smoke tests (require benchmark cells above to have been run) ──
print("\n=== Post-run comparator smoke tests ===\n")

_any_run = False
for _label, _r in [("SECOM", result_secom), ("Urban", result_urban), ("Graphene", result_graphene)]:
    if _r is None:
        continue
    _any_run = True

    _check(f"{_label}: robust_stability in [0, 1]",
           0.0 <= _r.get("robust_stability", -1.0) <= 1.0)
    _check(f"{_label}: comparator keys = {{anova, rfecv, boruta}}",
           {"anova", "rfecv", "boruta"} == set(_r.get("comparators", {}).keys()))

    _anova = _r["comparators"]["anova"]
    _check(f"{_label}: ANOVA is ComparatorResult",    isinstance(_anova, ComparatorResult))
    _check(f"{_label}: ANOVA score finite",           np.isfinite(_anova.mean_score))
    _check(f"{_label}: ANOVA stability in [0, 1]",   0.0 <= _anova.stability <= 1.0)
    _check(f"{_label}: ANOVA fold count = outer_cv", len(_anova.fold_scores) == bs._OUTER_CV)
    _check(f"{_label}: ANOVA feature names ⊆ original",
           all(f in set(_r["dataset"].X_train.columns) for f in _anova.fold_feature_sets[0]))

    _rfecv = _r["comparators"]["rfecv"]
    _check(f"{_label}: RFECV is ComparatorResult",   isinstance(_rfecv, ComparatorResult))
    _check(f"{_label}: RFECV score finite",          np.isfinite(_rfecv.mean_score))
    _check(f"{_label}: RFECV stability in [0, 1]",  0.0 <= _rfecv.stability <= 1.0)
    _check(f"{_label}: RFECV selects ≥ 1 feature",  all(len(s) >= 1 for s in _rfecv.fold_feature_sets))

    _boruta = _r["comparators"]["boruta"]
    if _boruta is not None:
        _check(f"{_label}: Boruta score finite",        np.isfinite(_boruta.mean_score))
        _check(f"{_label}: Boruta stability in [0, 1]", 0.0 <= _boruta.stability <= 1.0)
    else:
        print(f"  ⚠  {_label}: Boruta skipped — install with: pip install boruta")

    # Score-range sanity by task type
    _is_reg = _r["task_type"] == "regression"
    if _is_reg:
        _check(f"{_label}: ANOVA neg-RMSE ≤ 0",  _anova.mean_score <= 0.0)
        _check(f"{_label}: RFECV neg-RMSE ≤ 0",  _rfecv.mean_score <= 0.0)
    else:
        _check(f"{_label}: ANOVA AUC in [0, 1]", 0.0 <= _anova.mean_score <= 1.0)
        _check(f"{_label}: RFECV AUC in [0, 1]", 0.0 <= _rfecv.mean_score <= 1.0)

if not _any_run:
    print("  No benchmark results available yet — run the benchmark cells above first.")

print("\nAll checks complete.")

=== jaccard_stability() unit tests ===

  ✔  identical sets  → J = 1.0
  ✔  disjoint sets   → J = 0.0
  ✔  50% overlap     → J = 0.5
  ✔  three-way 1/3   → J ≈ 1/3
  ✔  single set      → NaN
  ✔  empty list      → NaN
  ✔  two empty sets  → J = 1.0

=== ComparatorResult structural tests ===

  ✔  mean_score correct
  ✔  std_score correct
  ✔  mean_n_features = 2.0
  ✔  stability in [0, 1]

=== Post-run comparator smoke tests ===

  ✔  SECOM: robust_stability in [0, 1]
  ✔  SECOM: comparator keys = {anova, rfecv, boruta}
  ✔  SECOM: ANOVA is ComparatorResult
  ✔  SECOM: ANOVA score finite
  ✔  SECOM: ANOVA stability in [0, 1]
  ✔  SECOM: ANOVA fold count = outer_cv
  ✔  SECOM: ANOVA feature names ⊆ original
  ✔  SECOM: RFECV is ComparatorResult
  ✔  SECOM: RFECV score finite
  ✔  SECOM: RFECV stability in [0, 1]
  ✔  SECOM: RFECV selects ≥ 1 feature
  ✔  SECOM: Boruta score finite
  ✔  SECOM: Boruta stability in [0, 1]
  ✔  SECOM: ANOVA AUC in [0, 1]
  ✔  SECOM: RFECV AUC in [0, 1]
  ✔ 

---
## Synthetic recovery benchmarks

Three synthetic scenarios with known ground truth, evaluated under the same nested CV configuration as the real-data benchmarks. The informative features are known by construction, so each selection method can be scored on **recovery**: precision, recall, and F1 against the true informative set, plus the correlate-confusion rate (how often a selector picks a correlated decoy instead of the true cause). This is the only operational definition of 'rightness' available without domain ground truth.


In [ ]:
ds_synb = bs._make_synthetic_binary()
print(f'Synthetic Binary: {ds_synb.X.shape[0]} samples x {ds_synb.X.shape[1]} features')
print(f'  Train: {ds_synb.X_train.shape[0]}  |  Test: {ds_synb.X_test.shape[0]}')
print(f'  Informative: {len(ds_synb.true_features)}  Correlated decoys: {len(ds_synb.correlate_features)}')
print(f'  Positive-class rate (train): {ds_synb.y_train.mean():.3f}')
result_synb = run_scenario(ds_synb, verbose=True)
bs.print_synthetic_recovery(result_synb)


In [ ]:
ds_synm = bs._make_synthetic_multiclass()
print(f'Synthetic Multiclass: {ds_synm.X.shape[0]} samples x {ds_synm.X.shape[1]} features')
print(f'  Train: {ds_synm.X_train.shape[0]}  |  Test: {ds_synm.X_test.shape[0]}')
print(f'  Informative: {len(ds_synm.true_features)}  Correlated decoys: {len(ds_synm.correlate_features)}')
print(f'  Classes: {sorted(ds_synm.y_train.unique())}')
result_synm = run_scenario(ds_synm, verbose=True)
bs.print_synthetic_recovery(result_synm)


In [ ]:
ds_synr = bs._make_synthetic_regression()
print(f'Synthetic Regression: {ds_synr.X.shape[0]} samples x {ds_synr.X.shape[1]} features')
print(f'  Train: {ds_synr.X_train.shape[0]}  |  Test: {ds_synr.X_test.shape[0]}')
print(f'  Informative: {len(ds_synr.true_features)}  Correlated decoys: {len(ds_synr.correlate_features)}')
print(f'  Target std: {ds_synr.y_train.std():.3f}')
result_synr = run_scenario(ds_synr, verbose=True)
bs.print_synthetic_recovery(result_synr)


---
## Inter-method consensus and Friedman cross-method test

Two cross-method analyses run on the already-fitted results above. **Inter-method consensus** asks whether the four selectors converge on the same features on the same fold. Features in the consensus intersection are more likely to be 'right' than features unique to a single method. The **Friedman test** is an omnibus paired comparison across all four methods on the per-fold predictive scores; it identifies whether any method is systematically different in rank, with pairwise Wilcoxon p-values reported as a robust substitute for Nemenyi.


In [ ]:
all_results = [r for r in [result_secom, result_urban, result_graphene,
                           result_synb, result_synm, result_synr] if r is not None]

for r in all_results:
    # Build per-method per-fold sets for this dataset, then run consensus
    method_sets = {'ROBUST': [set(s) for s in r['robust_result'].nested_cv_result.selected_features_per_fold]}
    for _, cres in (r.get('comparators') or {}).items():
        if cres is None:
            continue
        method_sets[cres.name] = [set(s) for s in cres.fold_feature_sets]
    cons = bs.intermethod_consensus(method_sets)
    print(f'\n>>> {r["name"]}')
    bs.print_intermethod_consensus(cons)


In [ ]:
import numpy as np

for r in all_results:
    is_reg = r['task_type'] == 'regression'
    # All method scores in the 'higher is better' convention; for regression this
    # is neg-RMSE, which is exactly what fold_scores already stores.
    method_scores = {'ROBUST': np.asarray(r['robust_result'].nested_cv_result.outer_scores, dtype=float),
                     'Baseline': np.asarray(r['baseline']['fold_scores'], dtype=float)}
    for _, cres in (r.get('comparators') or {}).items():
        if cres is None:
            continue
        method_scores[cres.name] = np.asarray(cres.fold_scores, dtype=float)
    test = bs.friedman_nemenyi(method_scores)
    print(f'\n>>> {r["name"]}')
    bs.print_cross_method_friedman(test)


---
## Recovery summary across the three synthetic scenarios

Consolidated precision / recall / F1 of each selector against the known informative feature set, averaged across the K = 10 outer folds. This is the cleanest test of whether a selector finds the **right** features (those that genuinely drive the response) as opposed to merely a stable or score-competitive subset.


In [ ]:
import pandas as pd, numpy as np

rows = []
for r in [result_synb, result_synm, result_synr]:
    if r is None:
        continue
    ds = r['dataset']
    true_set = set(ds.true_features)
    cor_set  = set(ds.correlate_features or [])
    def _agg(label, fold_sets):
        rec = bs.fold_recovery_metrics(fold_sets, true_set, cor_set)
        rows.append({
            'Dataset': ds.name, 'Method': label,
            'Precision': rec['precision_mean'], 'Recall': rec['recall_mean'], 'F1': rec['f1_mean'],
            'TP': rec['tp_mean'], 'FP': rec['fp_mean'], 'FN': rec['fn_mean'],
            'CorrPick': rec['correlate_picks_mean'], 'NoisePick': rec['noise_picks_mean'],
        })
    _agg('ROBUST', [set(s) for s in r['robust_result'].nested_cv_result.selected_features_per_fold])
    for _, cres in (r.get('comparators') or {}).items():
        if cres is None:
            continue
        _agg(cres.name, [set(s) for s in cres.fold_feature_sets])

recovery_df = pd.DataFrame(rows)
print('Recovery against ground truth (mean across 10 outer folds):')
print(recovery_df.to_string(index=False, formatters={
    'Precision': '{:.3f}'.format, 'Recall': '{:.3f}'.format, 'F1': '{:.3f}'.format,
    'TP': '{:.1f}'.format, 'FP': '{:.1f}'.format, 'FN': '{:.1f}'.format,
    'CorrPick': '{:.1f}'.format, 'NoisePick': '{:.1f}'.format,
}))
